In [0]:
# ============================================================
# NOTEBOOK 03 — PREDICTIVE MODEL
# Vermont School - Early Warning System V2
#
# TRAIN:  24-25 data (T1+T2 features → final risk level)
# PREDICT: 25-26 data (T1+T2 features → predicted risk level)
# ============================================================

import os
os.environ['SPARKML_TEMP_DFS_PATH'] = '/Volumes/workspace/vermont/silver/tmp'
dbutils.fs.mkdirs('/Volumes/workspace/vermont/silver/tmp')

TRUSTED = "/Volumes/workspace/vermont/trusted"
SILVER  = "/Volumes/workspace/vermont/silver"

TRAIN_PATH   = f"{TRUSTED}/train_dataset"
PREDICT_PATH = f"{TRUSTED}/predict_dataset"
MODEL_PATH   = f"{SILVER}/early_warning_model"
PRED_OUTPUT  = f"{SILVER}/predictions_25_26"

GROUPS = [
    'Science', 'I_and_S', 'Mathematics', 'English',
    'Lengua_Castellana', 'Mandarin', 'Financial_Maths',
    'ICT_STEM', 'Physical_Education', 'Research_Methodology'
]

FEATURES = (
    [f'{g}_T1' for g in GROUPS] +
    [f'{g}_T2' for g in GROUPS] +
    ['total_absences', 'absence_class', 'absence_school',
     'late', 'early_leave', 'n_f1', 'n_f2', 'n_fault_types']
)

from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
import pandas as pd
import numpy as np

# Load datasets
df_train_pd   = spark.read.parquet(TRAIN_PATH).toPandas()
df_predict_pd = spark.read.parquet(PREDICT_PATH).toPandas()

print(f"✓ Training set:   {len(df_train_pd)} students")
print(f"✓ Prediction set: {len(df_predict_pd)} students")
print(f"\nTarget distribution (training):")
print(df_train_pd['risk_level'].value_counts().to_string())

In [0]:
# CELDA 2 — Prepare Spark dataframes and class weights

# Fill NaN grades with subject mean (better than 0 for grades)
df_train_filled = df_train_pd.copy()
grade_features = [f'{g}_T1' for g in GROUPS] + [f'{g}_T2' for g in GROUPS]

for col in grade_features:
    if col in df_train_filled.columns:
        col_mean = df_train_filled[col].mean()
        df_train_filled[col] = df_train_filled[col].fillna(col_mean)

df_predict_filled = df_predict_pd.copy()
for col in grade_features:
    if col in df_predict_filled.columns:
        col_mean = df_train_filled[col].mean()  # use TRAIN mean, not predict
        df_predict_filled[col] = df_predict_filled[col].fillna(col_mean)

# Available features (intersection of defined and available)
available = [f for f in FEATURES if f in df_train_filled.columns]
print(f"Available features: {len(available)} of {len(FEATURES)}")
missing = [f for f in FEATURES if f not in df_train_filled.columns]
if missing:
    print(f"Missing: {missing}")

# Class weights
conteos = df_train_filled['risk_level'].value_counts()
n_total = len(df_train_filled)
n_classes = 3
pesos = {cls: round(n_total / (n_classes * cnt), 4) 
         for cls, cnt in conteos.items()}
print(f"\nClass weights:")
for k, v in pesos.items():
    print(f"  {k}: {v}")

# Add weight column
df_train_filled['class_weight'] = df_train_filled['risk_level'].map(pesos)

# Convert to Spark
df_train_spark = spark.createDataFrame(df_train_filled[available + ['risk_level', 'class_weight']])

print(f"\n✓ Training Spark DataFrame: {df_train_spark.count()} rows")
print(f"✓ Using TRAIN mean imputation for NaN grades")
print(f"✓ Test/evaluation: 5-Fold CV on 24-25 only")
print(f"✓ 25-26 used for deployment only (no evaluation)")

In [0]:
# CELDA 3 — Train Random Forest with 5-Fold CV

# String indexer for target
indexer = StringIndexer(
    inputCol="risk_level",
    outputCol="label",
    stringOrderType="alphabetAsc"
)

# Assemble features
assembler = VectorAssembler(
    inputCols=available,
    outputCol="features",
    handleInvalid="keep"
)

# Random Forest
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="class_weight",
    seed=42
)

# Pipeline
pipeline = Pipeline(stages=[indexer, assembler, rf])

# Grid search
paramGrid = ParamGridBuilder()\
    .addGrid(rf.numTrees, [50, 100, 150])\
    .addGrid(rf.maxDepth, [3, 5, 7])\
    .build()

# Evaluator — F1 score (prioritize recall on minority classes)
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

# 5-Fold Cross Validation
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=5,
    seed=42,
    parallelism=2
)

print("Training with 5-Fold Cross-Validation...")
print(f"  Grid: {len(paramGrid)} combinations × 5 folds = {len(paramGrid)*5} runs")
cv_model = cv.fit(df_train_spark)
print("✓ Training complete")

# Best model metrics
best_rf = cv_model.bestModel.stages[-1]
print(f"\n── Best hyperparameters ──")
print(f"  numTrees: {best_rf.getNumTrees}")
print(f"  maxDepth: {best_rf.getOrDefault('maxDepth')}")
print(f"\n── Cross-Validation Results ──")
print(f"  F1-Score (avg): {max(cv_model.avgMetrics):.4f}")

In [0]:
# CELDA 4 — Feature importance and predictions on 25-26

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Feature importance
importances = pd.DataFrame({
    'feature':    available,
    'importance': best_rf.featureImportances.toArray()
}).sort_values('importance', ascending=False)

print("── Top 15 most important features ──")
print(importances.head(15).to_string(index=False))

# Plot
fig, ax = plt.subplots(figsize=(10, 7))
top15 = importances.head(15)
colors = ['#e74c3c' if '_T1' in f or '_T2' in f else '#3498db' 
          for f in top15['feature']]
ax.barh(top15['feature'][::-1], top15['importance'][::-1], color=colors[::-1])
ax.set_title('Feature Importance — Early Warning Model V2\nTrained on 24-25 | T1+T2 features only', 
             fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('/tmp/feature_importance_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Plot saved")

# ── Predict on 25-26 ──
print("\nGenerating predictions for 25-26...")

# Prepare prediction dataframe
df_pred_spark = spark.createDataFrame(
    df_predict_filled[available + ['student_id','grade','section_anon','risk_level',
                                   'avg_cumulative','min_cumulative','n_subjects_below_4'] +
                      [c for c in df_predict_filled.columns if '_T3' in c]]
)

# Apply model
predictions = cv_model.bestModel.transform(df_pred_spark)

# Map label index back to class name
label_map = {float(i): label for i, label in 
             enumerate(cv_model.bestModel.stages[0].labels)}

predictions_pd = predictions.select(
    'student_id', 'grade', 'section_anon',
    'risk_level', 'prediction', 'probability',
    'avg_cumulative', 'min_cumulative', 'n_subjects_below_4',
    *[c for c in df_predict_filled.columns if '_T3' in c]
).toPandas()

predictions_pd['predicted_risk'] = predictions_pd['prediction'].map(label_map)
predictions_pd['confidence'] = predictions_pd['probability'].apply(
    lambda x: round(float(max(x)), 3)
)

# Agreement between model prediction and current risk level
predictions_pd['agrees_with_current'] = (
    predictions_pd['predicted_risk'] == predictions_pd['risk_level']
)

print(f"✓ Predictions generated: {len(predictions_pd)} students")
print(f"\n── Predicted risk distribution (25-26) ──")
print(predictions_pd['predicted_risk'].value_counts().to_string())
print(f"\n── Agreement with current risk level ──")
print(f"  Agree:    {predictions_pd['agrees_with_current'].sum()} students")
print(f"  Disagree: {(~predictions_pd['agrees_with_current']).sum()} students")
print(f"\n── Disagreements (interesting cases) ──")
disagree = predictions_pd[~predictions_pd['agrees_with_current']][
    ['student_id','grade','section_anon',
     'risk_level','predicted_risk','confidence']
].sort_values('confidence', ascending=False)
print(disagree.to_string(index=False))

# Save predictions to Silver
spark.createDataFrame(predictions_pd.drop(columns=['probability'])).write\
    .mode("overwrite").parquet(PRED_OUTPUT)
print(f"\n✓ Predictions saved: {PRED_OUTPUT}")

In [0]:
# CELDA 5 — COMPARACIÓN DE MODELOS DE CLASIFICACIÓN
# Vermont School - Early Warning System V2
#
# Modelos: Logística, Árbol, Random Forest, GBT, Voting Ensemble
# Métricas: PR-AUC, ROC-AUC, Balanced Acc, F1, Precision, Recall
# + Matriz de confusión + Curva ROC + Curva de ganancia acumulada

# ── Librerías ──
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, 
                               GradientBoostingClassifier,
                               VotingClassifier)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score, average_precision_score,
    confusion_matrix, classification_report,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize

print("✓ Librerías cargadas")

# ── Cargar datos enriquecidos ──
TRAIN_FE_PATH   = f"{TRUSTED}/train_dataset_fe"
PREDICT_FE_PATH = f"{TRUSTED}/predict_dataset_fe"

df_train = spark.read.parquet(TRAIN_FE_PATH).toPandas()
df_predict = spark.read.parquet(PREDICT_FE_PATH).toPandas()

print(f"✓ Train: {len(df_train)} estudiantes")
print(f"✓ Predict: {len(df_predict)} estudiantes")

# ── Features para clasificación ──
# Incluimos las nuevas features del feature engineering
FEATURES_CLASIF = (
    [f'{g}_T1' for g in GROUPS] +
    [f'{g}_T2' for g in GROUPS] +
    [f'{g}_delta' for g in GROUPS] +
    ['total_absences', 'absence_class', 'late', 'early_leave',
     'n_f1', 'n_f2', 'n_fault_types',
     'tendencia_general', 'avg_T1', 'avg_T2',
     'n_bajo_T1', 'n_bajo_T2', 'delta_materias_bajo',
     'dispersion_T2', 'ratio_ausencia_clase', 'indice_disciplinario',
     'min_nota_T2', 'min_nota_T1']
)

# Solo las que existen en el dataset
available_clasif = [f for f in FEATURES_CLASIF if f in df_train.columns]
print(f"\n✓ Features disponibles: {len(available_clasif)} de {len(FEATURES_CLASIF)}")

# ── Preparar X e y ──
X = df_train[available_clasif].copy()
y = df_train['risk_level'].copy()

# Imputar NaN con media de cada columna
X = X.fillna(X.mean())

# Codificar target
le = LabelEncoder()
y_enc = le.fit_transform(y)
classes = le.classes_
print(f"\n✓ Clases: {list(classes)}")
print(f"  Distribución: {dict(zip(classes, np.bincount(y_enc)))}")

# ── Class weights ──
n_total = len(y_enc)
n_classes = len(classes)
class_weights = {}
for i, cls in enumerate(classes):
    n_cls = np.sum(y_enc == i)
    class_weights[i] = round(n_total / (n_classes * n_cls), 4)
print(f"\n✓ Class weights: {dict(zip(classes, class_weights.values()))}")